In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from transformers import AutoModelForCausalLM
from tuned_lens import TunedLens

MODEL_NAME = "EleutherAI/pythia-160m-deduped"
DATA_DIR = Path("../token_evolution_data/excerpts")
N_LAYERS = 13  # layers 0-12: embedding + 12 transformer blocks
D_MODEL = 768

# Load model
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.float().eval()
final_ln = model.gpt_neox.final_layer_norm

# Load TunedLens (for Part 3: translated hidden states)
tuned_lens = TunedLens.from_model_and_pretrained(model, map_location="cpu").float()
tuned_lens.eval()
torch.set_grad_enabled(False)

print("Model and TunedLens loaded.")

In [ ]:
excerpt_files = sorted(DATA_DIR.glob("excerpt_*.npz"))
print(f"Found {len(excerpt_files)} excerpts")

categories = ["never", "by_4", "by_0"]
cat_labels = {
    "never": "Never stabilize",
    "by_4": "Stable by layer 4",
    "by_0": "Stable by layer 0",
}

# Accumulators for average cosine similarity matrices
cos_sum_raw = {c: np.zeros((N_LAYERS, N_LAYERS)) for c in categories}
cos_sum_tl  = {c: np.zeros((N_LAYERS, N_LAYERS)) for c in categories}
token_count = {c: 0 for c in categories}

# Reused later by the probe cell so we do not need a separate counting sweep.
probe_by0_total = 0


def accumulate_cos(h, acc):
    """Accumulate pairwise layer cosine similarities. h: (L, N, D), acc: (L, L)."""
    norms = np.linalg.norm(h, axis=2, keepdims=True) + 1e-10
    h_n = h / norms
    cos = np.einsum("ink,jnk->ijn", h_n, h_n)  # (L, L, N)
    acc += cos.sum(axis=2)


for fi, fpath in enumerate(excerpt_files):
    if fi % 200 == 0:
        print(f"  Processing {fi}/{len(excerpt_files)} ...")

    data = np.load(fpath, allow_pickle=True)
    hs = data["hidden_states"]          # (13, seq_len, 768)
    top_ids = data["tl_top_token_ids"]   # (13, seq_len)

    seq_len = hs.shape[1]
    end = seq_len - 1  # drop last token
    if end <= 0:
        continue

    # Writable float32 copy, dropping last token
    hs_work = np.array(hs[:, :end, :], dtype=np.float32)

    # NOTE: hidden_states[12] is already post-final-layer-norm (GPT-NeoX returns the
    # post-LN state as the last element of output_hidden_states). No need to apply
    # final_ln again.

    # Persistent top-1 layer (stabilization criterion)
    top_work = top_ids[:, :end]
    matches = top_work == top_work[-1:]
    cum = np.cumprod(matches[::-1], axis=0)[::-1]
    player = np.argmax(cum, axis=0)   # (end,)

    masks = {
        "never": player == 12,   # top-1 only matches at the final layer
        "by_4":  player <= 4,    # stabilizes at or before layer 4
        "by_0":  player == 0,    # top-1 never changes across depth
    }
    probe_by0_total += int(masks["by_0"].sum())

    # TunedLens translated hidden states
    hs_tl_arr = np.empty_like(hs_work)
    for l in range(12):
        h = torch.tensor(hs_work[l])  # layers 0-11 are un-modified raw states
        h_t = tuned_lens.transform_hidden(h, l)
        h_ln = tuned_lens.unembed.final_norm(h_t)
        hs_tl_arr[l] = h_ln.numpy()
    # Layer 12 is already post-final-LN, so just apply final_norm for consistency
    # (this is the identity-like operation since it's already been normed by the model)
    hs_tl_arr[12] = hs_work[12]

    # Accumulate cosine similarities by category
    for cat, mask in masks.items():
        n = int(mask.sum())
        if n == 0:
            continue
        token_count[cat] += n
        accumulate_cos(hs_work[:, mask, :],    cos_sum_raw[cat])
        accumulate_cos(hs_tl_arr[:, mask, :],  cos_sum_tl[cat])

# Compute mean cosine similarity matrices
cos_mean_raw = {c: cos_sum_raw[c] / max(token_count[c], 1) for c in categories}
cos_mean_tl  = {c: cos_sum_tl[c]  / max(token_count[c], 1) for c in categories}

print("\nToken counts per category:")
for c in categories:
    print(f"  {cat_labels[c]}: {token_count[c]:,}")
print(f"Probe target (all stable-by-0 tokens): {probe_by0_total:,}")
print("Done.")

In [ ]:

def plot_cossim_matrix(cos_mean_dict, categories, cat_labels, token_count):
    fig, axes = plt.subplots(1, 3, figsize=(8, 3), sharex=True, sharey=True)
    vmin, vmax = 0, 1

    for ax, c in zip(axes, categories):
        mat = cos_mean_dict[c].copy()
        mat[np.triu_indices(N_LAYERS, k=1)] = np.nan
        im = ax.imshow(mat.T, vmin=vmin, vmax=vmax, cmap="viridis", aspect="equal")
        ax.set_title(f"{cat_labels[c]}\n(n = {token_count[c]:,})")
        ax.set_xlabel("Layer")
        if ax is axes[0]:
            ax.set_ylabel("Layer")
        ax.set_xticks(range(0, N_LAYERS, 2))
        ax.set_yticks(range(0, N_LAYERS, 2))

    fig.colorbar(im, ax=axes, shrink=0.6, label="Cosine similarity", pad=0.05)
    return fig

fig2 = plot_cossim_matrix(cos_mean_raw, categories, cat_labels, token_count)
plt.savefig("../figures/raw_hidden_cossim_new.pdf", bbox_inches="tight")

This shows the cosine similarities between TunedLens "translated hidden states"

In [ ]:
fig3 = plot_cossim_matrix(cos_mean_tl, categories, cat_labels, token_count)
plt.savefig("../figures/translated_hidden_cossim.pdf", bbox_inches="tight")

In [ ]:
# Variant: combined 2x3 figure (rows = raw/translated; cols = baseline + deltas)
ref_cat = categories[0]  # leftmost baseline category (currently "never")
comp_cats = categories[1:]

row_specs = [
    ("Raw hidden states", cos_mean_raw),
    ("Translated hidden states", cos_mean_tl),
]

# Use one shared symmetric range for all delta panels.
delta_mats = []
for _, cos_dict in row_specs:
    base = cos_dict[ref_cat]
    for c in comp_cats:
        delta_mats.append(cos_dict[c] - base)

delta_absmax = max(float(np.nanmax(np.abs(d))) for d in delta_mats)
delta_absmax = max(delta_absmax, 1e-6)

fig, axes = plt.subplots(2, 3, figsize=(10, 6), sharex=True, sharey=True)
fig.subplots_adjust(left=0.08, right=0.98, top=0.95, bottom=0.22, wspace=-0.3, hspace=0.2)

for r, (_, cos_dict) in enumerate(row_specs):
    base_mat = cos_dict[ref_cat].copy()
    base_mat[np.triu_indices(N_LAYERS, k=1)] = np.nan
    im_abs = axes[r, 0].imshow(base_mat.T, vmin=0, vmax=1, cmap="viridis", aspect="equal")

    for j, c in enumerate(comp_cats, start=1):
        dmat = (cos_dict[c] - cos_dict[ref_cat]).copy()
        dmat[np.triu_indices(N_LAYERS, k=1)] = np.nan
        im_delta = axes[r, j].imshow(
            dmat.T,
            vmin=-delta_absmax,
            vmax=delta_absmax,
            cmap="RdBu_r",
            aspect="equal",
        )

    axes[r, 0].set_ylabel("Layer")

# Top-row column titles
axes[0, 0].set_title("Never stable")
axes[0, 1].set_title(r"Stable by $\ell = 4$")
axes[0, 2].set_title(r"Stable by $\ell = 0$")

for ax in axes.ravel():
    ax.set_xlabel("Layer")
    ax.set_xticks(range(0, N_LAYERS, 2))
    ax.set_yticks(range(0, N_LAYERS, 2))

# Add bold alphabetical panel labels: a), b), ...
for idx, ax in enumerate(axes.ravel()):
    ax.text(
        -0.05,
        1.12,
        f"{chr(97 + idx)})",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=12,
        fontweight="bold",
    )

# Horizontal colorbars centered below corresponding subplot groups
left_boxes = [axes[0, 0].get_position(), axes[1, 0].get_position()]
left_x0 = min(b.x0 for b in left_boxes)
left_x1 = max(b.x1 for b in left_boxes)
left_y0 = min(b.y0 for b in left_boxes)
left_w = (left_x1 - left_x0) * 0.82
left_x = left_x0 + (left_x1 - left_x0 - left_w) / 2
left_cax = fig.add_axes([left_x, left_y0 - 0.15, left_w, 0.025])
cb1 = fig.colorbar(im_abs, cax=left_cax, orientation="horizontal", extend='min')
cb1.set_label("Cosine similarity")

right_boxes = [axes[0, 1].get_position(), axes[0, 2].get_position(), axes[1, 1].get_position(), axes[1, 2].get_position()]
right_x0 = min(b.x0 for b in right_boxes)
right_x1 = max(b.x1 for b in right_boxes)
right_y0 = min(b.y0 for b in right_boxes)
right_w = (right_x1 - right_x0) * 0.65
right_x = right_x0 + (right_x1 - right_x0 - right_w) / 2
right_cax = fig.add_axes([right_x, right_y0 - 0.15, right_w, 0.025])
cb2 = fig.colorbar(im_delta, cax=right_cax, orientation="horizontal")
cb2.set_label(r"$\Delta$ cosine similarity")

plt.savefig("../figures/cossim_raw_translated_with_deltas.pdf", bbox_inches="tight")
plt.show()

### Cosine similarity by model cross-entropy decile

Instead of splitting tokens by TunedLens top-1 stabilization layer, split them by the model's own cross-entropy loss. We compare the bottom decile (lowest CE — easiest tokens) vs. the top decile (highest CE — hardest tokens).

In [ ]:
all_losses = []
for fi, fpath in enumerate(excerpt_files):
    data = np.load(fpath, allow_pickle=True)
    tl = data["token_losses"]  # (seq_len-1,)
    all_losses.append(tl)

all_losses = np.concatenate(all_losses)
print(f"Total tokens with losses: {len(all_losses):,}")
print(f"Loss range: [{all_losses.min():.4f}, {all_losses.max():.4f}]")

# Decile thresholds
lo_thresh = np.percentile(all_losses, 10)
hi_thresh = np.percentile(all_losses, 90)
print(f"Bottom decile (≤ {lo_thresh:.4f}): {(all_losses <= lo_thresh).sum():,} tokens")
print(f"Top decile    (≥ {hi_thresh:.4f}): {(all_losses >= hi_thresh).sum():,} tokens")

In [ ]:
# ── Pass 2: Accumulate cosine similarity matrices for CE deciles ──
ce_categories = ["low_ce", "high_ce"]
cos_sum_ce = {c: np.zeros((N_LAYERS, N_LAYERS)) for c in ce_categories}
token_count_ce = {c: 0 for c in ce_categories}

for fi, fpath in enumerate(excerpt_files):
    if fi % 200 == 0:
        print(f"  Processing {fi}/{len(excerpt_files)} ...")

    data = np.load(fpath, allow_pickle=True)
    hs = data["hidden_states"]       # (13, seq_len, 768)
    tl = data["token_losses"]        # (seq_len-1,)

    seq_len = hs.shape[1]
    end = seq_len - 1
    if end <= 0:
        continue

    hs_work = np.array(hs[:, :end, :], dtype=np.float32)

    # Masks based on model cross-entropy
    ce_masks = {
        "low_ce":  tl <= lo_thresh,
        "high_ce": tl >= hi_thresh,
    }

    for cat, mask in ce_masks.items():
        n = int(mask.sum())
        if n == 0:
            continue
        token_count_ce[cat] += n
        accumulate_cos(hs_work[:, mask, :], cos_sum_ce[cat])

cos_mean_ce = {c: cos_sum_ce[c] / max(token_count_ce[c], 1) for c in ce_categories}

print(f"\nToken counts:  low CE = {token_count_ce['low_ce']:,},  high CE = {token_count_ce['high_ce']:,}")
print("Done.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3), sharex=True, sharey=True)

mat_lo = cos_mean_ce["low_ce"].copy()
mat_hi = cos_mean_ce["high_ce"].copy()
mat_diff = mat_lo - mat_hi  # positive = low-CE tokens more similar

# Mask upper triangle for display
tri_mask = np.triu_indices(N_LAYERS, k=1)

# Left: Low CE decile
mat_show = mat_lo.copy()
mat_show[tri_mask] = np.nan
im0 = axes[0].imshow(mat_show.T, vmin=0, vmax=1, cmap="viridis", aspect="equal")
axes[0].set_title(f"Low CE (bottom 10%)\n(n = {token_count_ce['low_ce']:,})")
axes[0].set_ylabel("Layer")

# Middle: High CE decile
mat_show = mat_hi.copy()
mat_show[tri_mask] = np.nan
im1 = axes[1].imshow(mat_show.T, vmin=0, vmax=1, cmap="viridis", aspect="equal")
axes[1].set_title(f"High CE (top 10%)\n(n = {token_count_ce['high_ce']:,})")

# Colorbar for left/middle
fig.colorbar(im1, ax=axes[:2], shrink=0.6, label="Cosine similarity", pad=0.05)

# Right: Difference (Low - High)
mat_show = mat_diff.copy()
mat_show[tri_mask] = np.nan
im2 = axes[2].imshow(mat_show.T, vmin=-0.2, vmax=0.2, cmap="RdBu_r", aspect="equal")
axes[2].set_title("Difference\n(Low CE − High CE)")
fig.colorbar(im2, ax=axes[2], shrink=0.6, label="Δ Cosine similarity", pad=0.05)

for ax in axes:
    ax.set_xlabel("Layer")
    ax.set_xticks(range(0, N_LAYERS, 2))
    ax.set_yticks(range(0, N_LAYERS, 2))

plt.savefig("../figures/raw_hidden_cossim_by_ce_decile.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Recompute CE-decile cosine similarities with top-1 rogue dim removed ──
# Uses sorted_dims from the rogue dimension analysis (cell 13).
# sorted_dims[0] is the single dominant rogue dimension.

rogue_dim = sorted_dims[:1]
keep_dims = np.setdiff1d(np.arange(D_MODEL), rogue_dim)

cos_sum_ce_rm = {c: np.zeros((N_LAYERS, N_LAYERS)) for c in ce_categories}
token_count_ce_rm = {c: 0 for c in ce_categories}

for fi, fpath in enumerate(excerpt_files):
    if fi % 200 == 0:
        print(f"  Recomputing CE-decile (rogue removed) {fi}/{len(excerpt_files)} ...")

    data = np.load(fpath, allow_pickle=True)
    hs = data["hidden_states"]       # (13, seq_len, 768)
    tl = data["token_losses"]        # (seq_len-1,)

    seq_len = hs.shape[1]
    end = seq_len - 1
    if end <= 0:
        continue

    hs_work = np.array(hs[:, :end, :], dtype=np.float32)

    ce_masks = {
        "low_ce":  tl <= lo_thresh,
        "high_ce": tl >= hi_thresh,
    }

    for cat, mask in ce_masks.items():
        n = int(mask.sum())
        if n == 0:
            continue
        token_count_ce_rm[cat] += n
        accumulate_cos_subset(hs_work[:, mask, :], cos_sum_ce_rm[cat], keep_dims)

cos_mean_ce_rm = {c: cos_sum_ce_rm[c] / max(token_count_ce_rm[c], 1) for c in ce_categories}

print(f"\nToken counts (sanity):  low CE = {token_count_ce_rm['low_ce']:,},  high CE = {token_count_ce_rm['high_ce']:,}")
print(f"Rogue dimension removed: dim {rogue_dim[0]}")
print("Done.")

In [ ]:
# ── 3-panel figure: CE deciles with rogue dim removed ──
fig, axes = plt.subplots(1, 3, figsize=(10, 3), sharex=True, sharey=True)

mat_lo = cos_mean_ce_rm["low_ce"].copy()
mat_hi = cos_mean_ce_rm["high_ce"].copy()
mat_diff = mat_lo - mat_hi

tri_mask = np.triu_indices(N_LAYERS, k=1)

# Left: Low CE decile
mat_show = mat_lo.copy()
mat_show[tri_mask] = np.nan
im0 = axes[0].imshow(mat_show.T, vmin=0, vmax=1, cmap="viridis", aspect="equal")
axes[0].set_title(f"Low CE (bottom 10%)\n(n = {token_count_ce_rm['low_ce']:,})")
axes[0].set_ylabel("Layer")

# Middle: High CE decile
mat_show = mat_hi.copy()
mat_show[tri_mask] = np.nan
im1 = axes[1].imshow(mat_show.T, vmin=0, vmax=1, cmap="viridis", aspect="equal")
axes[1].set_title(f"High CE (top 10%)\n(n = {token_count_ce_rm['high_ce']:,})")

fig.colorbar(im1, ax=axes[:2], shrink=0.6, label="Cosine similarity", pad=0.05)

# Right: Difference
mat_show = mat_diff.copy()
mat_show[tri_mask] = np.nan
im2 = axes[2].imshow(mat_show.T, vmin=-0.2, vmax=0.2, cmap="RdBu_r", aspect="equal")
axes[2].set_title("Difference\n(Low CE − High CE)")
fig.colorbar(im2, ax=axes[2], shrink=0.6, label="Δ Cosine similarity", pad=0.05)

for ax in axes:
    ax.set_xlabel("Layer")
    ax.set_xticks(range(0, N_LAYERS, 2))
    ax.set_yticks(range(0, N_LAYERS, 2))

fig.suptitle(f"top-1 rogue dim removed (dim {rogue_dim[0]})", y=1.02)
plt.savefig("../figures/raw_hidden_cossim_by_ce_decile_rogue_removed.pdf", bbox_inches="tight")
plt.show()


For a subset of excerpts, we unembed each translated hidden state (apply the unembedding matrix) and check that the argmax matches `tl_top_token_ids` from the saved data.

In [ ]:
# Sanity check: unembed translated hidden states and verify top-1 match
unembed_W = tuned_lens.unembed.unembedding  # nn.Linear (vocab_size, hidden_dim)

N_CHECK = 100  # number of excerpts to check
check_files = sorted(DATA_DIR.glob("excerpt_*.npz"))[:N_CHECK]

total_tokens_checked = 0
matches_per_layer = np.zeros(N_LAYERS, dtype=np.int64)
total_per_layer = np.zeros(N_LAYERS, dtype=np.int64)

for fpath in check_files:
    data = np.load(fpath, allow_pickle=True)
    hs = data["hidden_states"]
    top_ids = data["tl_top_token_ids"]
    seq_len = hs.shape[1]
    end = seq_len - 1
    if end <= 0:
        continue

    hs_f32 = np.array(hs[:, :end, :], dtype=np.float32)
    top_ref = top_ids[:, :end] # (13, end)

    for l in range(12):
        h = torch.tensor(hs_f32[l])
        h_t = tuned_lens.transform_hidden(h, l)
        h_ln = tuned_lens.unembed.final_norm(h_t)
        logits = unembed_W(h_ln)  # (end, vocab_size)
        pred = logits.argmax(dim=-1).numpy()
        matches_per_layer[l] += (pred == top_ref[l]).sum()
        total_per_layer[l] += end

    # Final layer: hidden_states[12] is already post-final-layer-norm,
    # so unembed directly without applying final_norm again.
    h_final = torch.tensor(hs_f32[12])
    logits_final = unembed_W(h_final)
    pred_final = logits_final.argmax(dim=-1).numpy()
    matches_per_layer[12] += (pred_final == top_ref[12]).sum()
    total_per_layer[12] += end

    total_tokens_checked += end

acc = matches_per_layer / total_per_layer
print(f"Checked {N_CHECK} excerpts, {total_tokens_checked:,} tokens\n")
print("Layer | Top-1 match rate")
print("------+-----------------")
for l in range(N_LAYERS):
    print(f"  {l:2d}  |  {acc[l]:.6f}  ({matches_per_layer[l]}/{total_per_layer[l]})")

## Rogue dimensions analysis

We decompose the cosine similarity between layers into per-dimension contributions.
For a token with hidden states $h_i$ at layer $i$ and $h_j$ at layer $j$:

$$\cos(h_i, h_j) = \sum_d \frac{h_{i,d} \cdot h_{j,d}}{\|h_i\| \|h_j\|}$$

We compute the average contribution of each dimension to the inter-layer cosine similarity, then check whether a small number of "rogue dimensions" dominate.

In [ ]:
N_SAMPLE = 1600  # excerpts to sample for this analysis

# We compute average per-dimension contributions to inter-layer cosine similarity
global_dimwise_tl = np.zeros(D_MODEL)
global_dimwise_raw = np.zeros(D_MODEL)
global_count = 0

sample_files = sorted(DATA_DIR.glob("excerpt_*.npz"))[:N_SAMPLE]

for fi, fpath in enumerate(sample_files):
    if fi % 50 == 0:
        print(f"  Rogue dim analysis: {fi}/{N_SAMPLE} ...")

    data = np.load(fpath, allow_pickle=True)
    hs_raw = np.array(data["hidden_states"][:, :-1, :], dtype=np.float32)  # (13, end, 768)
    n_tok = hs_raw.shape[1]
    if n_tok <= 0:
        continue

    # Compute TunedLens translated states
    hs_tl = np.empty_like(hs_raw)
    for l in range(12):
        h_t = tuned_lens.transform_hidden(torch.tensor(hs_raw[l]), l)
        hs_tl[l] = tuned_lens.unembed.final_norm(h_t).numpy()
    hs_tl[12] = hs_raw[12]

    # For all layer pairs (i, j) with i < j:
    for i in range(N_LAYERS):
        for j in range(i + 1, N_LAYERS):
            # TL-transformed states
            norm_prod_tl = (
                np.linalg.norm(hs_tl[i], axis=1) * np.linalg.norm(hs_tl[j], axis=1) + 1e-10
            )  # (n_tok,)
            contrib_tl = (hs_tl[i] * hs_tl[j]) / norm_prod_tl[:, np.newaxis]  # (n_tok, 768)
            global_dimwise_tl += contrib_tl.mean(axis=0)

            # Raw hidden states
            norm_prod_raw = (
                np.linalg.norm(hs_raw[i], axis=1) * np.linalg.norm(hs_raw[j], axis=1) + 1e-10
            )  # (n_tok,)
            contrib_raw = (hs_raw[i] * hs_raw[j]) / norm_prod_raw[:, np.newaxis]  # (n_tok, 768)
            global_dimwise_raw += contrib_raw.mean(axis=0)

            global_count += 1

# Average across all layer pairs and excerpts
global_dimwise_tl /= global_count
global_dimwise_raw /= global_count

# Sort dimensions by average contribution
sorted_dims_tl = np.argsort(global_dimwise_tl)[::-1]
sorted_dims_raw = np.argsort(global_dimwise_raw)[::-1]

cumulative_tl = np.cumsum(global_dimwise_tl[sorted_dims_tl])
cumulative_raw = np.cumsum(global_dimwise_raw[sorted_dims_raw])
total_abs_tl = cumulative_tl[-1]
total_abs_raw = cumulative_raw[-1]

# Backward-compatible aliases used by downstream cells/plots
sorted_dims = sorted_dims_tl
global_dimwise = global_dimwise_tl
cumulative = cumulative_tl
total_abs = total_abs_tl

print("\nTop 10 dimensions by contribution to inter-layer cos sim (TunedLens translated states):")
print(f"{'Rank':>4}  {'Dim':>4}  {'Contribution':>14}  {'Cum. frac':>10}")
for rank in range(10):
    d = sorted_dims_tl[rank]
    print(f"{rank:4d}  {d:4d}  {global_dimwise_tl[d]:+14.6f}  {cumulative_tl[rank]/total_abs_tl:10.4f}")

print("\nTop 10 dimensions by contribution to inter-layer cos sim (raw hidden states):")
print(f"{'Rank':>4}  {'Dim':>4}  {'Contribution':>14}  {'Cum. frac':>10}")
for rank in range(10):
    d = sorted_dims_raw[rank]
    print(f"{rank:4d}  {d:4d}  {global_dimwise_raw[d]:+14.6f}  {cumulative_raw[rank]/total_abs_raw:10.4f}")

print("\nTop-1 rogue dimension index (average anisotropy contributor):")
print(f"  TunedLens transformed: {int(sorted_dims_tl[0])}")
print(f"  Raw hidden states:     {int(sorted_dims_raw[0])}")

print(f"\nTotal absolute contribution sum (TL):  {total_abs_tl:.6f}")
print(f"Total absolute contribution sum (raw): {total_abs_raw:.6f}")
print(f"Top-5 dims account for {cumulative_tl[4]/total_abs_tl:.1%} of TL total |contribution|")
print(f"Top-5 dims account for {cumulative_raw[4]/total_abs_raw:.1%} of raw total |contribution|")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3))

# (a) Bar chart of top-10 dimension contributions
top_k = 10
ax = axes[0]
ax.bar(np.arange(top_k)+1, global_dimwise[sorted_dims[:top_k]] / total_abs)
ax.set_xlabel("Dimension rank")
ax.set_xticks(np.arange(1, top_k+1))
ax.set_ylabel("Avg. contribution to cos sim")
ax.set_title(f"Top-{top_k} dimensions")
ax.axhline(0, c="gray", lw=0.5, ls="--")

# (b) Cumulative contribution
ax = axes[1]
ax.plot(cumulative / total_abs)
ax.set_ylim(0, 1.05)
ax.set_xlabel("Dimension (ranked)")
ax.set_title("Cumulative contribution")
ax.axhline(1.0, c="gray", lw=0.5, ls="--")

fig.tight_layout()
plt.savefig("../figures/rogue_dim_contributions.pdf", bbox_inches="tight")

## Recompute layer-wise cosine similarities with rogue dimensions removed

We remove the top-k rogue dimensions (k = 1, 3, 5) and recompute the translated-state cosine similarity matrices to see whether the lack of stabilization-dependence was an artifact of a few dominant dimensions.

In [ ]:
# ── Recompute cosine similarities with top-k rogue dims removed ──
K_VALUES = [3,]

# Precompute which dims to keep for each k (separately for TL and raw)
rogue_by_k_tl = {k: sorted_dims_tl[:k] for k in K_VALUES}
keep_by_k_tl  = {k: np.setdiff1d(np.arange(D_MODEL), rogue_by_k_tl[k]) for k in K_VALUES}

rogue_by_k_raw = {k: sorted_dims_raw[:k] for k in K_VALUES}
keep_by_k_raw  = {k: np.setdiff1d(np.arange(D_MODEL), rogue_by_k_raw[k]) for k in K_VALUES}

# Backward-compatible aliases
rogue_by_k = rogue_by_k_tl
keep_by_k = keep_by_k_tl

# Accumulators
cos_sum_tl_rm = {k: {c: np.zeros((N_LAYERS, N_LAYERS)) for c in categories} for k in K_VALUES}
cos_sum_raw_rm = {k: {c: np.zeros((N_LAYERS, N_LAYERS)) for c in categories} for k in K_VALUES}
token_count_rm = {c: 0 for c in categories}  # same mask counts shared by raw/TL

def accumulate_cos_subset(h, acc, dims):
    """Like accumulate_cos but only uses dimensions in `dims`."""
    h_sub = h[:, :, dims]  # (L, N, len(dims))
    norms = np.linalg.norm(h_sub, axis=2, keepdims=True) + 1e-10
    h_n = h_sub / norms
    cos = np.einsum("ink,jnk->ijn", h_n, h_n)
    acc += cos.sum(axis=2)


for fi, fpath in enumerate(excerpt_files):
    if fi % 200 == 0:
        print(f"  Recomputing ({fi}/{len(excerpt_files)}) ...")

    data = np.load(fpath, allow_pickle=True)
    hs = data["hidden_states"]
    top_ids = data["tl_top_token_ids"]
    seq_len = hs.shape[1]
    end = seq_len - 1
    if end <= 0:
        continue

    hs_work = np.array(hs[:, :end, :], dtype=np.float32)

    # Stabilization
    top_work = top_ids[:, :end]
    matches = top_work == top_work[-1:]
    cum = np.cumprod(matches[::-1], axis=0)[::-1]
    player = np.argmax(cum, axis=0)

    masks = {
        "never": player == 12,
        "by_4":  player <= 4,
        "by_0":  player == 0,
    }

    # TunedLens translated states
    hs_tl_arr = np.empty_like(hs_work)
    for l in range(12):
        h_t = tuned_lens.transform_hidden(torch.tensor(hs_work[l]), l)
        hs_tl_arr[l] = tuned_lens.unembed.final_norm(h_t).numpy()
    hs_tl_arr[12] = hs_work[12]  # already post-LN

    for cat, mask in masks.items():
        n = int(mask.sum())
        if n == 0:
            continue
        token_count_rm[cat] += n

        h_tl_masked = hs_tl_arr[:, mask, :]
        h_raw_masked = hs_work[:, mask, :]

        for k in K_VALUES:
            accumulate_cos_subset(h_tl_masked, cos_sum_tl_rm[k][cat], keep_by_k_tl[k])
            accumulate_cos_subset(h_raw_masked, cos_sum_raw_rm[k][cat], keep_by_k_raw[k])

cos_mean_tl_rm = {
    k: {c: cos_sum_tl_rm[k][c] / max(token_count_rm[c], 1) for c in categories}
    for k in K_VALUES
}
cos_mean_raw_rm = {
    k: {c: cos_sum_raw_rm[k][c] / max(token_count_rm[c], 1) for c in categories}
    for k in K_VALUES
}

print("\nDone. Token counts (sanity check):")
for c in categories:
    print(f"  {cat_labels[c]}: {token_count_rm[c]:,}")
for k in K_VALUES:
    print(f"k={k}: removed TL dim {int(rogue_by_k_tl[k][0])}, raw dim {int(rogue_by_k_raw[k][0])}")

Removing the rogue dimensions (there is really one dimension that is "dominant") from the cosine similarity computation tends to lower the cosine similarity across depth, which makes sense. However, there is still no difference in the hidden state similarity evolution among different stabilization depths.

In [ ]:
ref_cat = categories[0]
comp_cats = categories[1:]

row_specs = [
    ("Raw hidden states", cos_mean_raw_rm[3]),
    ("Translated hidden states", cos_mean_tl_rm[3]),
]

# Use one shared symmetric range for all delta panels.
delta_mats = []
for _, cos_dict in row_specs:
    base = cos_dict[ref_cat]
    for c in comp_cats:
        delta_mats.append(cos_dict[c] - base)

delta_absmax = max(float(np.nanmax(np.abs(d))) for d in delta_mats)
delta_absmax = max(delta_absmax, 1e-6)

fig, axes = plt.subplots(2, 3, figsize=(10, 6), sharex=True, sharey=True)
fig.subplots_adjust(left=0.08, right=0.98, top=0.95, bottom=0.22, wspace=-0.3, hspace=0.2)

for r, (_, cos_dict) in enumerate(row_specs):
    base_mat = cos_dict[ref_cat].copy()
    base_mat[np.triu_indices(N_LAYERS, k=1)] = np.nan
    im_abs = axes[r, 0].imshow(base_mat.T, vmin=0, vmax=1, cmap="viridis", aspect="equal")

    for j, c in enumerate(comp_cats, start=1):
        dmat = (cos_dict[c] - cos_dict[ref_cat]).copy()
        dmat[np.triu_indices(N_LAYERS, k=1)] = np.nan
        im_delta = axes[r, j].imshow(
            dmat.T,
            vmin=-delta_absmax,
            vmax=delta_absmax,
            cmap="RdBu_r",
            aspect="equal",
        )

    axes[r, 0].set_ylabel("Layer")

# Top-row column titles
axes[0, 0].set_title("Never stable")
axes[0, 1].set_title(r"Stable by $\ell = 4$")
axes[0, 2].set_title(r"Stable by $\ell = 0$")

for ax in axes.ravel():
    ax.set_xlabel("Layer")
    ax.set_xticks(range(0, N_LAYERS, 2))
    ax.set_yticks(range(0, N_LAYERS, 2))

# Add bold alphabetical panel labels: a), b), ...
for idx, ax in enumerate(axes.ravel()):
    ax.text(
        -0.05,
        1.12,
        f"{chr(97 + idx)})",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=12,
        fontweight="bold",
    )

# Horizontal colorbars centered below corresponding subplot groups
left_boxes = [axes[0, 0].get_position(), axes[1, 0].get_position()]
left_x0 = min(b.x0 for b in left_boxes)
left_x1 = max(b.x1 for b in left_boxes)
left_y0 = min(b.y0 for b in left_boxes)
left_w = (left_x1 - left_x0) * 0.82
left_x = left_x0 + (left_x1 - left_x0 - left_w) / 2
left_cax = fig.add_axes([left_x, left_y0 - 0.15, left_w, 0.025])
cb1 = fig.colorbar(im_abs, cax=left_cax, orientation="horizontal", extend='min')
cb1.set_label("Cosine similarity")

right_boxes = [axes[0, 1].get_position(), axes[0, 2].get_position(), axes[1, 1].get_position(), axes[1, 2].get_position()]
right_x0 = min(b.x0 for b in right_boxes)
right_x1 = max(b.x1 for b in right_boxes)
right_y0 = min(b.y0 for b in right_boxes)
right_w = (right_x1 - right_x0) * 0.65
right_x = right_x0 + (right_x1 - right_x0 - right_w) / 2
right_cax = fig.add_axes([right_x, right_y0 - 0.15, right_w, 0.025])
cb2 = fig.colorbar(im_delta, cax=right_cax, orientation="horizontal")
cb2.set_label(r"$\Delta$ cosine similarity")

plt.savefig("../figures/cossim_raw_translated_with_deltas_rogue_removed.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── H4: Aggregate token statistics by stabilization category ──
# With baseline normalization: we compute P(cat | token) / P(cat) to find tokens
# that are *disproportionately* in each category relative to their overall frequency.
from collections import Counter
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Accumulators
loss_by_cat = {c: [] for c in categories}
entropy_by_cat = {c: [] for c in categories}       # final-layer TunedLens entropy
token_id_counts = {c: Counter() for c in categories}
global_token_counts = Counter()  # baseline frequency across all tokens

for fi, fpath in enumerate(excerpt_files):
    if fi % 200 == 0:
        print(f"  Token stats: {fi}/{len(excerpt_files)} ...")

    data = np.load(fpath, allow_pickle=True)
    top_ids = data["tl_top_token_ids"]    # (13, seq_len)
    token_losses = data["token_losses"]   # (seq_len-1,)
    input_ids = data["input_ids"]         # (seq_len,)
    tl_entropy = data["tl_entropy"]       # (13, seq_len)

    seq_len = top_ids.shape[1]
    end = seq_len - 1
    if end <= 0:
        continue

    # Stabilization (same as cell 2)
    top_work = top_ids[:, :end]
    matches = top_work == top_work[-1:]
    cum_match = np.cumprod(matches[::-1], axis=0)[::-1]
    player = np.argmax(cum_match, axis=0)

    masks = {
        "never": player == 12,
        "by_4":  player <= 4,
        "by_0":  player == 0,
    }

    # Count ALL target tokens for baseline
    for i in range(end):
        tid = int(input_ids[i + 1])
        global_token_counts[tid] += 1

    for cat, mask in masks.items():
        idx = np.where(mask)[0]
        if len(idx) == 0:
            continue
        loss_by_cat[cat].extend(token_losses[idx].tolist())
        entropy_by_cat[cat].extend(tl_entropy[12, idx].tolist())
        for i in idx:
            tid = int(input_ids[i + 1])
            token_id_counts[cat][tid] += 1

# ── Compute normalized over/under-representation ──
total_tokens = sum(global_token_counts.values())

print("\n" + "="*70)
print("TOKEN STATISTICS BY STABILIZATION CATEGORY")
print("="*70)

for c in categories:
    losses = np.array(loss_by_cat[c])
    entropies = np.array(entropy_by_cat[c])
    n_cat = len(losses)
    p_cat = n_cat / total_tokens  # P(cat)
    print(f"\n── {cat_labels[c]} (n = {n_cat:,}, {p_cat:.1%} of all tokens) ──")
    print(f"  Cross-entropy loss:  mean={losses.mean():.3f}, median={np.median(losses):.3f}, "
          f"std={losses.std():.3f}")
    print(f"  Final-layer entropy: mean={entropies.mean():.3f}, median={np.median(entropies):.3f}")

    # Top-10 by raw count
    print(f"\n  Top-10 target tokens (by raw count):")
    print(f"    {'Token':>15s}  {'Count':>6s}  {'% in cat':>8s}  {'% overall':>10s}  {'Lift':>6s}")
    for tok_id, count in token_id_counts[c].most_common(10):
        tok_str = tokenizer.decode([tok_id])
        pct_cat = count / n_cat * 100
        pct_global = global_token_counts[tok_id] / total_tokens * 100
        lift = (count / n_cat) / (global_token_counts[tok_id] / total_tokens)
        print(f"    {repr(tok_str):>15s}  {count:6d}  {pct_cat:7.1f}%  {pct_global:9.1f}%  {lift:6.2f}x")

    # Top-10 by lift (over-representation), filtering to tokens with >= 20 occurrences in cat
    print(f"\n  Top-10 target tokens (by lift = over-representation vs baseline):")
    print(f"    {'Token':>15s}  {'Count':>6s}  {'% in cat':>8s}  {'% overall':>10s}  {'Lift':>6s}")
    lifts = []
    for tok_id, count in token_id_counts[c].items():
        if count < 20:
            continue
        lift = (count / n_cat) / (global_token_counts[tok_id] / total_tokens)
        lifts.append((tok_id, count, lift))
    lifts.sort(key=lambda x: -x[2])
    for tok_id, count, lift in lifts[:10]:
        tok_str = tokenizer.decode([tok_id])
        pct_cat = count / n_cat * 100
        pct_global = global_token_counts[tok_id] / total_tokens * 100
        print(f"    {repr(tok_str):>15s}  {count:6d}  {pct_cat:7.1f}%  {pct_global:9.1f}%  {lift:6.2f}x")

print("\nDone.")

In [ ]:
# Build a balanced hidden-state probe dataset
# Positive class: all stable-by-0 tokens
# Negative class: same number of never-stabilize tokens (sampled)

rng = np.random.default_rng(0)
target_by0 = int(probe_by0_total)

if target_by0 <= 0:
    raise ValueError("probe_by0_total is zero. Run Cell 2 first to compute stabilization counts.")

# Float16 keeps memory manageable for full 13x768 trajectories; we cast to float32 at training time.
X_by0_raw = np.empty((target_by0, N_LAYERS, D_MODEL), dtype=np.float16)
X_never_raw = np.empty((target_by0, N_LAYERS, D_MODEL), dtype=np.float16)

by0_ptr = 0
never_seen = 0
never_kept = 0

for fi, fpath in enumerate(excerpt_files):
    if fi % 200 == 0:
        print(f"  Building balanced probe set: {fi}/{len(excerpt_files)} ...")

    data = np.load(fpath, allow_pickle=True)
    hs = data["hidden_states"]
    top_ids = data["tl_top_token_ids"]
    seq_len = hs.shape[1]
    end = seq_len - 1
    if end <= 0:
        continue

    hs_f16 = np.asarray(hs[:, :end, :], dtype=np.float16)

    top_work = top_ids[:, :end]
    matches = top_work == top_work[-1:]
    cum_match = np.cumprod(matches[::-1], axis=0)[::-1]
    player = np.argmax(cum_match, axis=0)

    by0_idx = np.where(player == 0)[0]
    never_idx = np.where(player == 12)[0]

    if by0_ptr < target_by0 and len(by0_idx) > 0:
        n_take = min(len(by0_idx), target_by0 - by0_ptr)
        X_by0_raw[by0_ptr : by0_ptr + n_take] = np.transpose(hs_f16[:, by0_idx[:n_take], :], (1, 0, 2))
        by0_ptr += n_take

    for idx in never_idx:
        traj = hs_f16[:, idx, :]
        never_seen += 1
        if never_kept < target_by0:
            X_never_raw[never_kept] = traj
            never_kept += 1
        else:
            j = int(rng.integers(never_seen))
            if j < target_by0:
                X_never_raw[j] = traj

if by0_ptr != target_by0:
    raise RuntimeError(f"Expected {target_by0} stable-by-0 tokens, collected {by0_ptr}.")
if never_kept < target_by0:
    raise RuntimeError(
        f"Not enough never-stabilize tokens for balancing: need {target_by0}, got {never_kept}."
    )

print("\nBalanced probe dataset ready:")
print(f"  Stable-by-0 tokens (all): {X_by0_raw.shape[0]:,}")
print(f"  Never-stabilize tokens (sampled): {X_never_raw.shape[0]:,}")
print(f"  Hidden-state tensor shape per class: {tuple(X_by0_raw.shape)}")

In [ ]:
# Can we predict stabilization from raw hidden state?
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from tqdm import tqdm

X_never = X_never_raw.astype(np.float32, copy=False)
X_by0 = X_by0_raw.astype(np.float32, copy=False)

probe_acc = np.zeros((N_LAYERS, 5))

for layer in tqdm(range(N_LAYERS)):
    X = np.concatenate([X_never[:, layer, :], X_by0[:, layer, :]], axis=0)
    y = np.concatenate([np.zeros(len(X_never), dtype=np.int64), np.ones(len(X_by0), dtype=np.int64)])

    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, C=1.0, random_state=0),
    )
    scores = cross_val_score(clf, X, y, cv=5, scoring="accuracy", n_jobs=-1)
    probe_acc[layer, :] = scores

In [ ]:
# Plot mean accuracy with fold range
layers = np.arange(N_LAYERS)
probe_mean = probe_acc.mean(axis=1)
probe_min = probe_acc.min(axis=1)
probe_max = probe_acc.max(axis=1)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(layers, probe_mean, "o-", markersize=5, label="Per-layer probe")
ax.fill_between(layers, probe_min, probe_max, alpha=0.2, label="Fold range")
ax.axhline(0.5, ls=":", c="gray", label="Chance")
ax.set_xlabel("Layer")
ax.set_ylabel("5-fold CV accuracy")
ax.set_title("Linear probe: never-stabilize vs stable-by-0")
ax.legend(fontsize=8, loc='upper left')
ax.set_xticks(layers)
ax.set_ylim(0.45, 1.0)
fig.tight_layout()
plt.savefig("../figures/linear_probe_stabilization_raw_hidden.pdf", bbox_inches="tight")
plt.show()